# Backtest Baseline â€” Prediction Engine V2.2

**Purpose**: Establish frozen baseline metrics for the V2.2 prediction engine (commit 43c306c).  
All subsequent changes to `matchup_predictor.py` or `elo_lookup.py` must produce metrics â‰¥ these numbers on held-out data before merging.

**Improvements validated here:**
- QW-2: Stage 3 uses `z_stuff - z_power` (pitcher stuff vs batter power)
- QW-3: Career/season ELO blend for cold-start players
- ME-3: Home-field logit shift to Stage 2 hit probability
- LR-1: Calibrated zscore divisors (temporal generalization test)
- LR-2: Dynamic 2B/3B/HR ratios by speed/power ELO

**Deferred (require game-level outcome data):**
- QW-1: Pitcher clutch â†’ SP win probability (needs W/L game outcomes)
- QW-4: Fangraphs IP per start (needs per-pitcher IP actuals from Statcast)
- ME-1: OHLC form (requires `talent_daily_ohlc` per-date join, add separately)

In [1]:
import os
import sys
import math
from datetime import date, timedelta
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from dotenv import load_dotenv
from supabase import create_client

# Add project root so src/ is importable
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Reuse functions from backtest.py
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts'))
from backtest import (
    OUTCOME_MAP, OUTCOMES,
    compute_log_loss, compute_brier_scores,
    compute_calibration, compute_fantasy_point_error,
)
from src.fantasy.matchup_predictor import predict_plate_appearance, MLB_ELO_DISTRIBUTION
from src.fantasy.fantasy_calculator import estimate_batter_points, load_scoring_config

load_dotenv()
supabase = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_KEY'])
scoring = load_scoring_config()
print('Connected to Supabase')

Connected to Supabase


## Section 1 â€” Data Loading

Load plate appearances (including `home_team`/`away_team` for ME-3) and player ELOs  
(including `event_count`, `speed`, `clutch` for QW-3/LR-2 analysis).

In [2]:
SAMPLE_LIMIT = 300_000
PAGE_SIZE = 1_000  # Supabase PostgREST max rows per request

# Paginate via .range() because Supabase caps single responses at 1,000 rows
all_rows = []
offset = 0
while len(all_rows) < SAMPLE_LIMIT:
    resp = (
        supabase.table('plate_appearances')
        .select('pa_id, batter_id, pitcher_id, game_date, result_type, home_team, away_team')
        .not_.is_('batter_id', 'null')
        .not_.is_('pitcher_id', 'null')
        .not_.is_('game_date', 'null')
        .order('game_date', desc=False)
        .range(offset, offset + PAGE_SIZE - 1)
        .execute()
    )
    page = resp.data or []
    if not page:
        break
    all_rows.extend(page)
    offset += len(page)
    if len(page) < PAGE_SIZE:
        break  # last page

pa_df = pd.DataFrame(all_rows[:SAMPLE_LIMIT])
pa_df['outcome'] = pa_df['result_type'].map(OUTCOME_MAP)
pa_df = pa_df.dropna(subset=['outcome'])
pa_df['game_date'] = pd.to_datetime(pa_df['game_date']).dt.date

print(f'Loaded {len(pa_df):,} plate appearances')
print(f'Date range: {pa_df["game_date"].min()} → {pa_df["game_date"].max()}')
print(f'Weeks covered: {(pa_df["game_date"].max() - pa_df["game_date"].min()).days // 7}')
pa_df['outcome'].value_counts(normalize=True).round(4)

INFO HTTP Request: GET https://tbwbzzlyrecneqxgklgf.supabase.co/rest/v1/plate_appearances?select=pa_id%2Cbatter_id%2Cpitcher_id%2Cgame_date%2Cresult_type%2Chome_team%2Caway_team&batter_id=not.is.null&pitcher_id=not.is.null&game_date=not.is.null&order=game_date.asc&offset=0&limit=1000 "HTTP/2 200 OK"


INFO HTTP Request: GET https://tbwbzzlyrecneqxgklgf.supabase.co/rest/v1/plate_appearances?select=pa_id%2Cbatter_id%2Cpitcher_id%2Cgame_date%2Cresult_type%2Chome_team%2Caway_team&batter_id=not.is.null&pitcher_id=not.is.null&game_date=not.is.null&order=game_date.asc&offset=1000&limit=1000 "HTTP/2 200 OK"


INFO HTTP Request: GET https://tbwbzzlyrecneqxgklgf.supabase.co/rest/v1/plate_appearances?select=pa_id%2Cbatter_id%2Cpitcher_id%2Cgame_date%2Cresult_type%2Chome_team%2Caway_team&batter_id=not.is.null&pitcher_id=not.is.null&game_date=not.is.null&order=game_date.asc&offset=2000&limit=1000 "HTTP/2 200 OK"


INFO HTTP Request: GET https://tbwbzzlyrecneqxgklgf.supabase.co/rest/v1/plate_appearances?select=pa_id%2Cbatter_id%2Cpitcher_id%2Cgame_date%2Cresult_type%2Chome_team%2Caway_team&batter_id=not.is.null&pitcher_id=not.is.null&game_date=not.is.null&order=game_date.asc&offset=3000&limit=1000 "HTTP/2 200 OK"


INFO HTTP Request: GET https://tbwbzzlyrecneqxgklgf.supabase.co/rest/v1/plate_appearances?select=pa_id%2Cbatter_id%2Cpitcher_id%2Cgame_date%2Cresult_type%2Chome_team%2Caway_team&batter_id=not.is.null&pitcher_id=not.is.null&game_date=not.is.null&order=game_date.asc&offset=4000&limit=1000 "HTTP/2 200 OK"


...

INFO HTTP Request: GET https://tbwbzzlyrecneqxgklgf.supabase.co/rest/v1/plate_appearances?select=pa_id%2Cbatter_id%2Cpitcher_id%2Cgame_date%2Cresult_type%2Chome_team%2Caway_team&batter_id=not.is.null&pitcher_id=not.is.null&game_date=not.is.null&order=game_date.asc&offset=198000&limit=1000 "HTTP/2 200 OK"


INFO HTTP Request: GET https://tbwbzzlyrecneqxgklgf.supabase.co/rest/v1/plate_appearances?select=pa_id%2Cbatter_id%2Cpitcher_id%2Cgame_date%2Cresult_type%2Chome_team%2Caway_team&batter_id=not.is.null&pitcher_id=not.is.null&game_date=not.is.null&order=game_date.asc&offset=199000&limit=1000 "HTTP/2 200 OK"


INFO HTTP Request: GET https://tbwbzzlyrecneqxgklgf.supabase.co/rest/v1/plate_appearances?select=pa_id%2Cbatter_id%2Cpitcher_id%2Cgame_date%2Cresult_type%2Chome_team%2Caway_team&batter_id=not.is.null&pitcher_id=not.is.null&game_date=not.is.null&order=game_date.asc&offset=200000&limit=1000 "HTTP/2 200 OK"


Loaded 200,751 plate appearances
Date range: 2025-03-27 → 2026-04-12
Weeks covered: 54


outcome
OUT    0.4635
K      0.2223
1B     0.1420
BB     0.0853
2B     0.0422
HR     0.0304
HBP    0.0110
3B     0.0034
Name: proportion, dtype: float64

In [3]:
all_ids = list(set(pa_df['batter_id'].tolist() + pa_df['pitcher_id'].tolist()))

# Load full ELO record including event_count, career_elo, all talent dimensions
elo_rows = []
batch_size = 200
for i in range(0, len(all_ids), batch_size):
    batch = all_ids[i:i + batch_size]
    resp = (
        supabase.table('talent_player_current')
        .select('player_id, player_role, talent_type, season_elo, career_elo, event_count')
        .in_('player_id', batch)
        .execute()
    )
    elo_rows.extend(resp.data or [])

elo_df = pd.DataFrame(elo_rows)

# Defaults
DEF_B = {'contact': 1504.5, 'power': 1468.6, 'discipline': 1700.3, 'speed': 1500.0, 'clutch': 1500.0}
DEF_P = {'stuff': 1587.3, 'bip_suppression': 1513.3, 'command': 1681.1, 'clutch': 1500.0}

def build_player_dicts(df):
    """Build per-player ELO dicts keyed by player_id."""
    batter_elos, pitcher_elos = {}, {}
    batter_events, pitcher_events = defaultdict(dict), defaultdict(dict)
    for _, row in df.iterrows():
        pid = row['player_id']
        dim = row['talent_type']
        elo = float(row['season_elo'] or 1500.0)
        n = int(row['event_count'] or 0)
        if row['player_role'] == 'batter':
            batter_elos.setdefault(pid, {})[dim] = elo
            batter_events[pid][dim] = n
        else:
            pitcher_elos.setdefault(pid, {})[dim] = elo
            pitcher_events[pid][dim] = n
    return batter_elos, pitcher_elos, dict(batter_events), dict(pitcher_events)

batter_elos, pitcher_elos, batter_events, pitcher_events = build_player_dicts(elo_df)

# Also load batter team for ME-3 home/away analysis
player_resp = (
    supabase.table('players')
    .select('player_id, team')
    .in_('player_id', list(set(pa_df['batter_id'].tolist())))
    .execute()
)
batter_team = {r['player_id']: r['team'] for r in (player_resp.data or [])}

print(f'ELO data: {len(batter_elos):,} batters, {len(pitcher_elos):,} pitchers')
print(f'Team data: {len(batter_team):,} batters with team info')

INFO HTTP Request: GET https://tbwbzzlyrecneqxgklgf.supabase.co/rest/v1/talent_player_current?select=player_id%2Cplayer_role%2Ctalent_type%2Cseason_elo%2Ccareer_elo%2Cevent_count&player_id=in.%28663554%2C663556%2C663558%2C663559%2C663562%2C688138%2C622608%2C663568%2C663574%2C688158%2C663584%2C663586%2C663604%2C663609%2C663616%2C622663%2C663624%2C663623%2C663647%2C606303%2C663656%2C663662%2C663671%2C663687%2C647304%2C663697%2C663698%2C647315%2C647336%2C622761%2C688297%2C663728%2C671922%2C663731%2C680118%2C647351%2C663738%2C622780%2C663743%2C622786%2C434378%2C663757%2C663765%2C663767%2C663773%2C671976%2C688363%2C663795%2C663796%2C663804%2C606466%2C672012%2C672016%2C663837%2C663853%2C663855%2C663878%2C663886%2C663893%2C663897%2C663898%2C663903%2C688497%2C663941%2C663947%2C663967%2C663968%2C663969%2C663978%2C663992%2C663993%2C664023%2C664034%2C664040%2C664056%2C664059%2C664062%2C688642%2C664068%2C664074%2C664076%2C655889%2C672275%2C672279%2C680474%2C672282%2C672284%2C623149%2C664123%2C6641

INFO HTTP Request: GET https://tbwbzzlyrecneqxgklgf.supabase.co/rest/v1/talent_player_current?select=player_id%2Cplayer_role%2Ctalent_type%2Cseason_elo%2Ccareer_elo%2Cevent_count&player_id=in.%28607259%2C656413%2C656420%2C656427%2C500779%2C681006%2C689200%2C672820%2C689225%2C672841%2C656457%2C656458%2C656464%2C656484%2C689254%2C656492%2C689266%2C681082%2C664702%2C689296%2C664728%2C656537%2C664731%2C656541%2C656546%2C656550%2C664744%2C656555%2C664747%2C656557%2C664761%2C681146%2C681151%2C672960%2C656577%2C664770%2C664774%2C656582%2C681168%2C656605%2C607455%2C681190%2C681198%2C656629%2C607481%2C656638%2C656641%2C681217%2C689414%2C664849%2C615698%2C664854%2C664875%2C656686%2C607536%2C656716%2C681293%2C681297%2C664913%2C656730%2C656731%2C689520%2C664948%2C623993%2C664954%2C681343%2C681347%2C681351%2C656775%2C607625%2C664983%2C656794%2C664991%2C656811%2C681393%2C476594%2C681402%2C665019%2C607680%2C640448%2C640451%2C640454%2C640455%2C640459%2C640462%2C656848%2C656849%2C697812%2C673237%2C6814

INFO HTTP Request: GET https://tbwbzzlyrecneqxgklgf.supabase.co/rest/v1/talent_player_current?select=player_id%2Cplayer_role%2Ctalent_type%2Cseason_elo%2Ccareer_elo%2Cevent_count&player_id=in.%28682144%2C542881%2C657571%2C542888%2C673962%2C673965%2C657585%2C682171%2C682177%2C665795%2C682183%2C665804%2C657612%2C690382%2C542932%2C592094%2C665828%2C665833%2C665839%2C657649%2C542963%2C682227%2C657656%2C518397%2C682243%2C805123%2C665861%2C665862%2C682254%2C665871%2C665877%2C641302%2C674072%2C592155%2C502043%2C641312%2C641313%2C682274%2C813349%2C502054%2C641329%2C592178%2C608566%2C641343%2C592192%2C665923%2C502085%2C665926%2C641355%2C592206%2C543056%2C657746%2C518489%2C657756%2C657757%2C665953%2C641386%2C665966%2C690544%2C641401%2C543101%2C534910%2C592254%2C805249%2C608650%2C608665%2C608671%2C543135%2C666018%2C666023%2C805299%2C805300%2C575929%2C518585%2C608701%2C518595%2C641482%2C592332%2C608717%2C608718%2C641487%2C805326%2C608723%2C592351%2C674285%2C805367%2C805373%2C641540%2C666120%2C5432

INFO HTTP Request: GET https://tbwbzzlyrecneqxgklgf.supabase.co/rest/v1/talent_player_current?select=player_id%2Cplayer_role%2Ctalent_type%2Cseason_elo%2Ccareer_elo%2Cevent_count&player_id=in.%28682842%2C445276%2C682847%2C682848%2C666464%2C519008%2C543592%2C682868%2C682877%2C592773%2C641927%2C592779%2C641933%2C502671%2C805779%2C641941%2C691094%2C641943%2C592791%2C682927%2C682928%2C805808%2C805811%2C592836%2C642016%2C592866%2C642020%2C519141%2C691172%2C691176%2C682985%2C625643%2C682988%2C682989%2C691182%2C519151%2C682990%2C691185%2C682995%2C592885%2C682998%2C682997%2C683002%2C666619%2C683004%2C609280%2C666624%2C642048%2C683011%2C683021%2C543760%2C805904%2C674841%2C666659%2C666661%2C642086%2C691251%2C642100%2C543807%2C642121%2C519242%2C683083%2C691277%2C683090%2C642133%2C666711%2C642136%2C650333%2C642152%2C543859%2C674944%2C642180%2C543877%2C683146%2C683155%2C519317%2C650391%2C642201%2C519326%2C642207%2C650402%2C642215%2C683175%2C642216%2C691373%2C666808%2C642232%2C642239%2C691406%2C6832

INFO HTTP Request: GET https://tbwbzzlyrecneqxgklgf.supabase.co/rest/v1/talent_player_current?select=player_id%2Cplayer_role%2Ctalent_type%2Cseason_elo%2Ccareer_elo%2Cevent_count&player_id=in.%28675911%2C675915%2C675919%2C667755%2C643217%2C675989%2C676044%2C676051%2C643289%2C676059%2C676070%2C676083%2C676092%2C700669%2C643338%2C676106%2C676116%2C684320%2C643361%2C676130%2C700712%2C643376%2C643377%2C643396%2C643410%2C545121%2C692585%2C676206%2C643446%2C676254%2C676263%2C676272%2C643511%2C676282%2C455117%2C455119%2C643565%2C700932%2C676356%2C676391%2C676395%2C545341%2C668227%2C676428%2C545361%2C676439%2C676440%2C676466%2C676467%2C676475%2C676477%2C676480%2C594580%2C676508%2C676510%2C676534%2C701121%2C660162%2C676551%2C676571%2C676572%2C668390%2C676596%2C676604%2C676609%2C676617%2C807712%2C807713%2C660271%2C676661%2C676664%2C676679%2C676680%2C676684%2C676694%2C676701%2C676702%2C471911%2C594798%2C676724%2C807799%2C701305%2C578428%2C676742%2C553869%2C676755%2C676760%2C553882%2C701350%2C6767

INFO HTTP Request: GET https://tbwbzzlyrecneqxgklgf.supabase.co/rest/v1/talent_player_current?select=player_id%2Cplayer_role%2Ctalent_type%2Cseason_elo%2Ccareer_elo%2Cevent_count&player_id=in.%28693686%2C554430%2C554431%2C669127%2C669134%2C669145%2C521692%2C677347%2C669160%2C669193%2C669194%2C669199%2C669200%2C669203%2C669208%2C669211%2C669212%2C472610%2C669221%2C669224%2C669234%2C669236%2C800311%2C669242%2C693821%2C669256%2C669257%2C669261%2C669270%2C669276%2C628317%2C693855%2C669288%2C669289%2C702056%2C669302%2C702070%2C669304%2C669308%2C669326%2C669330%2C489119%2C669357%2C669358%2C685744%2C448179%2C669364%2C669369%2C669371%2C669372%2C669373%2C669374%2C669387%2C669391%2C669392%2C669394%2C677587%2C677588%2C669397%2C669398%2C677592%2C677594%2C677595%2C702176%2C628451%2C628452%2C685801%2C669422%2C702193%2C669432%2C669438%2C579328%2C669449%2C669450%2C702222%2C669456%2C677649%2C669459%2C677651%2C669461%2C694037%2C669467%2C669477%2C595751%2C595777%2C702284%2C702303%2C702332%2C661388%2C7023

INFO HTTP Request: GET https://tbwbzzlyrecneqxgklgf.supabase.co/rest/v1/talent_player_current?select=player_id%2Cplayer_role%2Ctalent_type%2Cseason_elo%2Ccareer_elo%2Cevent_count&player_id=in.%28702795%2C670032%2C678225%2C678226%2C670036%2C670042%2C678246%2C670059%2C547180%2C547179%2C571760%2C547184%2C801139%2C686452%2C694646%2C571771%2C686469%2C686475%2C670092%2C694671%2C670097%2C670102%2C678316%2C686527%2C801216%2C694728%2C670156%2C694738%2C670167%2C686554%2C686555%2C621020%2C670174%2C678368%2C686563%2C621028%2C670183%2C621035%2C621043%2C686580%2C678391%2C678394%2C621051%2C621053%2C621057%2C571912%2C670223%2C670224%2C686610%2C686611%2C686613%2C670231%2C571927%2C670242%2C694819%2C670245%2C571945%2C571946%2C571948%2C686642%2C621107%2C621111%2C621112%2C621114%2C686654%2C621121%2C571970%2C694851%2C670276%2C506433%2C670280%2C686668%2C621139%2C686676%2C686681%2C678489%2C678495%2C686701%2C670329%2C662139%2C801403%2C694918%2C686730%2C621199%2C678554%2C686747%2C801434%2C686751%2C686752%2C6867

INFO HTTP Request: GET https://tbwbzzlyrecneqxgklgf.supabase.co/rest/v1/talent_player_current?select=player_id%2Cplayer_role%2Ctalent_type%2Cseason_elo%2Ccareer_elo%2Cevent_count&player_id=in.%28605463%2C695578%2C687396%2C687401%2C605483%2C695600%2C605488%2C687424%2C605507%2C605513%2C671056%2C630105%2C802139%2C605540%2C687462%2C695657%2C671083%2C687473%2C687478%2C695670%2C671096%2C695681%2C671106%2C671109%2C671111%2C687515%2C671131%2C687529%2C695734%2C671162%2C679358%2C687551%2C687589%2C671212%2C671213%2C687597%2C671218%2C671221%2C687606%2C687637%2C572955%2C548384%2C671277%2C671284%2C671286%2C671289%2C671305%2C573009%2C679529%2C802415%2C802419%2C663158%2C687765%2C450203%2C622253%2C622268%2C573124%2C679631%2C687830%2C696030%2C687847%2C687849%2C687859%2C687863%2C573186%2C687888%2C573204%2C663321%2C663330%2C614179%2C696100%2C687911%2C622379%2C687922%2C687931%2C663362%2C696131%2C663368%2C696136%2C663372%2C573262%2C687952%2C696147%2C687957%2C696149%2C679775%2C663399%2C802686%2C663423%2C6634

INFO HTTP Request: GET https://tbwbzzlyrecneqxgklgf.supabase.co/rest/v1/players?select=player_id%2Cteam&player_id=in.%28692225%2C624641%2C681987%2C669701%2C595978%2C669707%2C808975%2C669717%2C667670%2C808982%2C669720%2C669722%2C663584%2C663586%2C667690%2C669743%2C596019%2C663604%2C677941%2C677942%2C677943%2C571448%2C663609%2C677950%2C677951%2C663616%2C608324%2C677956%2C663624%2C571466%2C675915%2C456781%2C694359%2C694362%2C608348%2C663647%2C694374%2C663656%2C694376%2C694377%2C663662%2C694384%2C608369%2C690291%2C694388%2C678009%2C678011%2C608385%2C596103%2C647304%2C686217%2C665742%2C643217%2C663697%2C596115%2C663698%2C657557%2C665750%2C596117%2C702616%2C645277%2C622761%2C673962%2C596142%2C663728%2C596146%2C663731%2C645302%2C680118%2C647351%2C645305%2C663743%2C682177%2C682183%2C665804%2C663757%2C676044%2C542932%2C669911%2C643289%2C676059%2C694497%2C665828%2C676070%2C671976%2C665833%2C688363%2C665839%2C542963%2C663796%2C657656%2C606466%2C665861%2C665862%2C571657%2C672012%2C672016%2C676116%

ELO data: 793 batters, 913 pitchers
Team data: 693 batters with team info


## Section 2 â€” Run Predictions

Run `predict_plate_appearance` for each PA row, passing `speed_elo`, `clutch_elo`, and `is_home`.

In [4]:
def get_batter(pid):
    d = batter_elos.get(pid, {})
    return {
        'contact': d.get('contact', DEF_B['contact']),
        'power': d.get('power', DEF_B['power']),
        'discipline': d.get('discipline', DEF_B['discipline']),
    }

def get_pitcher(pid):
    d = pitcher_elos.get(pid, {})
    return {
        'stuff': d.get('stuff', DEF_P['stuff']),
        'bip_suppression': d.get('bip_suppression', DEF_P['bip_suppression']),
        'command': d.get('command', DEF_P['command']),
    }

pred_probs = []
pred_meta = []  # parallel list with z_diffs and stage values

for _, row in pa_df.iterrows():
    bid, pid = row['batter_id'], row['pitcher_id']
    b = get_batter(bid)
    p = get_pitcher(pid)
    speed_elo = batter_elos.get(bid, {}).get('speed', DEF_B['speed'])
    clutch_elo = batter_elos.get(bid, {}).get('clutch', DEF_B['clutch'])
    batter_team_code = batter_team.get(bid)
    is_home = batter_team_code is not None and batter_team_code == row.get('home_team')
    result = predict_plate_appearance(b, p, clutch_elo=clutch_elo, is_home=is_home, speed_elo=speed_elo)
    pred_probs.append(result['probabilities'])
    pred_meta.append({
        **result['z_diffs'],
        **result['stages']['stage3'],
        'is_home': is_home,
        'power_elo': batter_elos.get(bid, {}).get('power', DEF_B['power']),
        'speed_elo': speed_elo,
        'clutch_elo': clutch_elo,
        'contact_events': batter_events.get(bid, {}).get('contact', 0),
        'power_events': batter_events.get(bid, {}).get('power', 0),
    })

meta_df = pd.DataFrame(pred_meta)
outcomes = pa_df['outcome'].tolist()
print(f'Predictions complete: {len(pred_probs):,} PAs')
print(f'Home PAs: {meta_df["is_home"].sum():,} ({meta_df["is_home"].mean():.1%})')

Predictions complete: 200,751 PAs
Home PAs: 0 (0.0%)


## Section 3 â€” Baseline Metrics

**These numbers are the frozen baseline.** Future changes must beat or match all of them.

In [5]:
ll = compute_log_loss(pred_probs, outcomes)
naive_ll = -math.log(1 / len(OUTCOMES))
brier = compute_brier_scores(pred_probs, outcomes)

print(f'=== Log-Loss ===')
print(f'Model:  {ll:.5f}')
print(f'Naive:  {naive_ll:.5f}  (uniform over {len(OUTCOMES)} outcomes)')
print(f'Lift:   {naive_ll - ll:+.5f}')
print()
print('=== Brier Scores per Outcome (lower = better) ===')
brier_df = pd.DataFrame([
    {'outcome': o, 'brier': s, 'actual_rate': outcomes.count(o) / len(outcomes)}
    for o, s in sorted(brier.items())
])
print(brier_df.to_string(index=False))

=== Log-Loss ===
Model:  1.47969
Naive:  2.07944  (uniform over 8 outcomes)
Lift:   +0.59975

=== Brier Scores per Outcome (lower = better) ===
outcome    brier  actual_rate
     1B 0.121603     0.142027
     2B 0.040374     0.042172
     3B 0.003376     0.003387
     BB 0.077776     0.085255
    HBP 0.010900     0.011024
     HR 0.029400     0.030351
      K 0.171461     0.222285
    OUT 0.248161     0.463500


In [6]:
# Calibration decile tables for HR and BB (the two most important outcomes)
for outcome in ['HR', 'BB']:
    cal = compute_calibration(pred_probs, outcomes, outcome, n_bins=10)
    print(f'\nCalibration for P({outcome}) â€” decile bins:')
    print(cal.to_string(index=False))


Calibration for P(HR) â€” decile bins:
             bin  mean_predicted  mean_actual  count
(0.0091, 0.0193]        0.017476     0.020322  20077
(0.0193, 0.0212]        0.020351     0.023613  20074
(0.0212, 0.0227]        0.021987     0.025953  20075
(0.0227, 0.0241]        0.023401     0.026052  20075
(0.0241, 0.0254]        0.024750     0.028692  20075
(0.0254, 0.0268]        0.026109     0.030235  20076
(0.0268, 0.0284]        0.027573     0.029837  20076
(0.0284, 0.0302]        0.029255     0.033876  20073
 (0.0302, 0.033]        0.031482     0.037908  20075
 (0.033, 0.0501]        0.036168     0.047024  20075

Calibration for P(BB) â€” decile bins:
             bin  mean_predicted  mean_actual  count
(0.0447, 0.0668]        0.062891     0.060620  20076
(0.0668, 0.0707]        0.068881     0.064907  20075
(0.0707, 0.0737]        0.072229     0.070535  20075
(0.0737, 0.0763]        0.074999     0.076758  20076
(0.0763, 0.0788]        0.077552     0.079107  20074
(0.0788, 0.0814]   

C:\Users\Jake\Documents\Python\Baseball\fantasy-matchup-predictor\scripts\backtest.py:177: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return df.groupby('bin').agg(
C:\Users\Jake\Documents\Python\Baseball\fantasy-matchup-predictor\scripts\backtest.py:177: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return df.groupby('bin').agg(


In [7]:
# Per-player weekly fantasy point error and Spearman rank correlation
weekly = compute_fantasy_point_error(pa_df, pred_probs, scoring)
mae = weekly['abs_error'].mean()
rmse = (weekly['error'] ** 2).mean() ** 0.5

# Only include player-weeks with >= 5 PAs for reliable rank comparison
rho_df = weekly[weekly['pa_count'] >= 5].copy()
rho, pval = spearmanr(rho_df['pred_pts'], rho_df['actual_pts'])

print(f'=== Weekly Fantasy Point Error (per player-week) ===')
print(f'MAE:              {mae:.2f} pts')
print(f'RMSE:             {rmse:.2f} pts')
print(f'Player-weeks:     {len(weekly):,}')
print(f'Player-weeks â‰¥5PA:{len(rho_df):,}')
print()
print(f'=== Rank Correlation (Spearman) ===')
print(f'rho:    {rho:.4f}  (target > 0.60)')
print(f'p-val:  {pval:.4f}')

=== Weekly Fantasy Point Error (per player-week) ===
MAE:              4.38 pts
RMSE:             5.70 pts
Player-weeks:     12,038
Player-weeks â‰¥5PA:10,959

=== Rank Correlation (Spearman) ===
rho:    0.6000  (target > 0.60)
p-val:  0.0000


## Section 4 â€” Per-Improvement Validation

In [8]:
# --- QW-2: Stage 3 Pitcher-Power ---
# Split matchups by z_stuff_power quartile; compare predicted HR rate vs actual.
# If QW-2 is calibrated, HR rate should increase monotonically from Q1â†’Q4 (lower z_stuff_power = batter dominates).

qw2 = pa_df[['outcome']].copy()
qw2['z_stuff_power'] = meta_df['z_stuff_power'].values
qw2['pred_hr'] = [p['HR'] for p in pred_probs]
qw2['actual_hr'] = (qw2['outcome'] == 'HR').astype(float)
qw2['quartile'] = pd.qcut(qw2['z_stuff_power'], q=4, labels=['Q1 (batter dominates)', 'Q2', 'Q3', 'Q4 (pitcher dominates)'])

qw2_summary = qw2.groupby('quartile', observed=True).agg(
    pred_hr_rate=('pred_hr', 'mean'),
    actual_hr_rate=('actual_hr', 'mean'),
    count=('actual_hr', 'count'),
).reset_index()
qw2_summary['error'] = qw2_summary['pred_hr_rate'] - qw2_summary['actual_hr_rate']

print('QW-2: HR rate by z_stuff_power quartile')
print('Expected: pred_hr_rate decreases Q1â†’Q4 (pitcher dominates = fewer HRs)')
print(qw2_summary.to_string(index=False))

QW-2: HR rate by z_stuff_power quartile
Expected: pred_hr_rate decreases Q1â†’Q4 (pitcher dominates = fewer HRs)
              quartile  pred_hr_rate  actual_hr_rate  count     error
 Q1 (batter dominates)      0.032433        0.041821  50190 -0.009388
                    Q2      0.027143        0.031543  50186 -0.004400
                    Q3      0.023904        0.026003  50187 -0.002099
Q4 (pitcher dominates)      0.019940        0.022037  50188 -0.002097


In [9]:
# --- QW-3: Career/Season ELO Blend (Cold-Start Analysis) ---
# Split batters by event_count relative to reliability thresholds:
#   contact/discipline: 400 PA; power: 200 PA
# Use the minimum (200) as the global cold-start cutoff.
# Cold-start batters should have lower MAE because the blend pulls them toward
# their career average instead of noisy small-sample season ELO.

COLD_THRESHOLD = 200  # power reliability threshold (most restrictive)

pa_with_events = pa_df.copy()
pa_with_events['power_events'] = pa_df['batter_id'].map(
    lambda bid: batter_events.get(bid, {}).get('power', 0)
)
pa_with_events['cold_start'] = pa_with_events['power_events'] < COLD_THRESHOLD

# Per-PA actual and predicted points
outcome_pts = {
    'BB': 1.0, 'HBP': 1.0, 'K': -1.0, 'OUT': 0.0,
    '1B': 1 + 0.40 + 0.45,
    '2B': 2 + 0.40 + 0.45,
    '3B': 3 + 0.40 + 0.45,
    'HR': 4 + 0.40 + 0.45,
}
pa_with_events['actual_pts'] = pa_with_events['outcome'].map(outcome_pts).fillna(0.0)
pa_with_events['pred_pts'] = [estimate_batter_points(p, scoring, pas=1) for p in pred_probs]
pa_with_events['abs_error'] = (pa_with_events['pred_pts'] - pa_with_events['actual_pts']).abs()

# Aggregate to weekly level per batter
pa_with_events['week_start'] = pa_with_events['game_date'].apply(
    lambda d: d - timedelta(days=d.weekday())
)
weekly_qw3 = pa_with_events.groupby(['batter_id', 'week_start']).agg(
    abs_error=('abs_error', 'sum'),
    cold_start=('cold_start', 'first'),
    pa_count=('abs_error', 'count'),
).reset_index()

cold_mae = weekly_qw3[weekly_qw3['cold_start']]['abs_error'].mean()
warm_mae = weekly_qw3[~weekly_qw3['cold_start']]['abs_error'].mean()
cold_n = weekly_qw3['cold_start'].sum()
warm_n = (~weekly_qw3['cold_start']).sum()

print('QW-3: Cold-start vs warm-start batter weekly MAE')
print(f'  Cold-start (<{COLD_THRESHOLD} power PAs): MAE={cold_mae:.2f} pts  (n={cold_n:,} player-weeks)')
print(f'  Warm-start (â‰¥{COLD_THRESHOLD} power PAs): MAE={warm_mae:.2f} pts  (n={warm_n:,} player-weeks)')
ratio = cold_mae / warm_mae if warm_mae > 0 else float('nan')
print(f'  Ratio (cold/warm): {ratio:.3f}  â€” flag if > 1.15')
if ratio > 1.15:
    print(f'  WARNING: Cold-start MAE is {ratio:.1%} worse than warm-start; QW-3 blend may need tuning')

QW-3: Cold-start vs warm-start batter weekly MAE
  Cold-start (<200 power PAs): MAE=11.48 pts  (n=5,291 player-weeks)
  Warm-start (â‰¥200 power PAs): MAE=21.83 pts  (n=6,747 player-weeks)
  Ratio (cold/warm): 0.526  â€” flag if > 1.15


In [10]:
# --- ME-3: Home/Away Prediction Error ---
# After the home logit shift, home game PA outcomes should show higher actual
# hit rates for home batters matching our +0.010 logit shift.

pa_me3 = pa_df.copy()
pa_me3['is_home'] = meta_df['is_home'].values
pa_me3['pred_hit'] = [p['1B'] + p['2B'] + p['3B'] + p['HR'] for p in pred_probs]
pa_me3['actual_hit'] = pa_me3['outcome'].isin(['1B', '2B', '3B', 'HR']).astype(float)
pa_me3['hit_error'] = pa_me3['pred_hit'] - pa_me3['actual_hit']
pa_me3['abs_hit_error'] = pa_me3['hit_error'].abs()

def _stats(df):
    if df.empty:
        return {'actual_hit_rate': float('nan'), 'pred_hit_rate': float('nan'), 'mae': float('nan'), 'n': 0}
    return {
        'actual_hit_rate': df['actual_hit'].mean(),
        'pred_hit_rate': df['pred_hit'].mean(),
        'mae': df['abs_hit_error'].mean(),
        'n': len(df),
    }

home_stats = _stats(pa_me3[pa_me3['is_home']])
away_stats = _stats(pa_me3[~pa_me3['is_home']])

print('ME-3: Home vs Away hit prediction')
print(f'  Home: actual={home_stats["actual_hit_rate"]:.4f}  pred={home_stats["pred_hit_rate"]:.4f}  MAE={home_stats["mae"]:.4f}  n={home_stats["n"]:,}')
print(f'  Away: actual={away_stats["actual_hit_rate"]:.4f}  pred={away_stats["pred_hit_rate"]:.4f}  MAE={away_stats["mae"]:.4f}  n={away_stats["n"]:,}')
if home_stats['n'] < 100:
    print('  NOTE: Fewer than 100 home PAs identified — batter team data may be incomplete')

ME-3: Home vs Away hit prediction
  Home: actual=nan  pred=nan  MAE=nan  n=0
  Away: actual=0.2179  pred=0.2132  MAE=0.3373  n=200,751
  NOTE: Fewer than 100 home PAs identified — batter team data may be incomplete


In [11]:
# --- LR-1: Divisor Calibration (Temporal Generalization) ---
# 80/20 temporal split: evaluate log-loss on held-out 20% (most recent PAs).
# Compare current calibrated divisors vs the BEST single flat divisor
# (found by grid search on the train split — not an arbitrary 4.0).

# numpy already imported as np
import src.fantasy.matchup_predictor as mp_module

split_idx = int(len(pred_probs) * 0.80)
test_probs = pred_probs[split_idx:]
test_outcomes = outcomes[split_idx:]
test_rows = pa_df.iloc[split_idx:].reset_index(drop=True)
train_rows = pa_df.iloc[:split_idx].reset_index(drop=True)
train_outcomes_list = outcomes[:split_idx]

ll_calibrated = compute_log_loss(test_probs, test_outcomes)
original_divisors = dict(mp_module.ZSCORE_DIVISOR)

def _flat_ll(d, rows, out_list):
    """Log-loss under a single flat divisor, using the real predict_plate_appearance path."""
    mp_module.ZSCORE_DIVISOR.update({'stage1_bb': d, 'stage1_k': d, 'stage2': d, 'stage3': d})
    probs = [predict_plate_appearance(get_batter(r['batter_id']),
                                      get_pitcher(r['pitcher_id']))['probabilities']
             for _, r in rows.iterrows()]
    mp_module.ZSCORE_DIVISOR.update(original_divisors)
    return compute_log_loss(probs, out_list)

# Grid search on train split to find optimal flat divisor
grid = np.arange(2.0, 10.5, 0.5)
best_flat_d = float(min(grid, key=lambda d: _flat_ll(d, train_rows, train_outcomes_list)))

# Evaluate best flat on held-out test split
mp_module.ZSCORE_DIVISOR.update({'stage1_bb': best_flat_d, 'stage1_k': best_flat_d,
                                  'stage2': best_flat_d, 'stage3': best_flat_d, 'stage1_hbp': 7.0})
flat_probs_test = [predict_plate_appearance(get_batter(r['batter_id']),
                                             get_pitcher(r['pitcher_id']))['probabilities']
                   for _, r in test_rows.iterrows()]
ll_flat = compute_log_loss(flat_probs_test, test_outcomes)
mp_module.ZSCORE_DIVISOR.update(original_divisors)

print(f'LR-1: Held-out 20% log-loss ({len(test_probs):,} PAs)')
print(f'  Calibrated divisors:         {ll_calibrated:.5f}')
print(f'  Best flat divisor ({best_flat_d}):  {ll_flat:.5f}')
print(f'  Delta: {ll_calibrated - ll_flat:+.5f}  (negative = calibrated is better)')
if ll_calibrated < ll_flat:
    print('  PASS')
else:
    print('  FAIL — run calibrate_divisors.ipynb to update config')

LR-1: Held-out 20% log-loss (40,151 PAs)
  Calibrated divisors:         1.47145
  Best flat divisor (10.0):  1.47363
  Delta: -0.00218  (negative = calibrated is better)
  PASS


In [12]:
# --- LR-2: XBH Split Model (Dynamic 2B/3B/HR Ratios) ---
# Split batters by power ELO quartile.
# Expected: Q4 (highest power) should have higher actual HR% and higher predicted HR ratio.

lr2 = pa_df[['outcome']].copy()
lr2['power_elo'] = meta_df['power_elo'].values
lr2['pred_hr_ratio'] = meta_df['hr_ratio'].values
lr2['pred_3b_ratio'] = meta_df['3b_ratio'].values
lr2['speed_elo'] = meta_df['speed_elo'].values

# Only consider XBH PAs for ratio comparison
xbh_mask = lr2['outcome'].isin(['2B', '3B', 'HR'])
lr2_xbh = lr2[xbh_mask].copy()
lr2_xbh['actual_hr'] = (lr2_xbh['outcome'] == 'HR').astype(float)
lr2_xbh['actual_3b'] = (lr2_xbh['outcome'] == '3B').astype(float)
lr2_xbh['actual_2b'] = (lr2_xbh['outcome'] == '2B').astype(float)
lr2_xbh['power_quartile'] = pd.qcut(lr2_xbh['power_elo'], q=4, labels=['Q1 (lowest)', 'Q2', 'Q3', 'Q4 (highest)'])

lr2_summary = lr2_xbh.groupby('power_quartile', observed=True).agg(
    pred_hr_ratio=('pred_hr_ratio', 'mean'),
    actual_hr_rate=('actual_hr', 'mean'),
    pred_3b_ratio=('pred_3b_ratio', 'mean'),
    actual_3b_rate=('actual_3b', 'mean'),
    count=('actual_hr', 'count'),
).reset_index()

print('LR-2: XBH ratio by batter power quartile (among XBH PAs only)')
print('Expected: pred_hr_ratio and actual_hr_rate both increase Q1â†’Q4')
print(lr2_summary.to_string(index=False))

LR-2: XBH ratio by batter power quartile (among XBH PAs only)
Expected: pred_hr_ratio and actual_hr_rate both increase Q1â†’Q4
power_quartile  pred_hr_ratio  actual_hr_rate  pred_3b_ratio  actual_3b_rate  count
   Q1 (lowest)       0.340544        0.368325       0.049273        0.051047   3820
            Q2       0.366064        0.399637       0.047822        0.039938   3856
            Q3       0.383280        0.379952       0.046599        0.042010   3761
  Q4 (highest)       0.406909        0.451341       0.044258        0.045502   3802


## Section 5 â€” Red Flag Checks

Assertion checks for known failure modes. Investigate any failures â€” do not suppress.

In [13]:
import warnings

passed = []
failed = []

# Flag 1: BB rate > 15% for any batter — domain guard against runaway predictions.
#   Threshold: 15% is above any observed MLB BB rate; catches broken calibration, not extreme matchups.
#   (Prior 0.12 was set on un-normalized model; after V2.2 normalization fix extreme discipline/low-command
#   matchups legitimately reach 13-14%, well within baseball reality.)
high_bb = [(i, p['BB']) for i, p in enumerate(pred_probs) if p['BB'] > 0.15]
if high_bb:
    failed.append(f'BB rate > 15%: {len(high_bb):,} PAs (max={max(x[1] for x in high_bb):.4f})')
else:
    passed.append('BB rate <= 15% for all PAs')

# Flag 2: HR rate > 10% for any PA (Stage 3 divisor=5.0 may be too small for top-power batters)
high_hr = [(i, p['HR']) for i, p in enumerate(pred_probs) if p['HR'] > 0.10]
if high_hr:
    failed.append(f'HR rate > 10%: {len(high_hr):,} PAs (max={max(x[1] for x in high_hr):.4f})')
else:
    passed.append('HR rate â‰¤ 10% for all PAs')

# Flag 3: Probabilities must sum to ~1.0 for all PAs
bad_sum = [i for i, p in enumerate(pred_probs) if abs(sum(p.values()) - 1.0) > 0.001]
if bad_sum:
    failed.append(f'Probabilities do not sum to 1.0: {len(bad_sum):,} PAs')
else:
    passed.append('All PA probability sums â‰ˆ 1.0')

# Flag 4: Cold-start MAE should not be >15% worse than warm-start
if ratio > 1.15:
    failed.append(f'Cold-start MAE {ratio:.1%} worse than warm-start (threshold: 15%)')
else:
    passed.append(f'Cold-start MAE ratio OK ({ratio:.2f})')

# Flag 5: Model must beat naive baseline (uniform log-loss)
if ll >= naive_ll:
    failed.append(f'Model log-loss ({ll:.5f}) â‰¥ naive baseline ({naive_ll:.5f})')
else:
    passed.append(f'Model beats naive baseline by {naive_ll - ll:.5f}')

print('=== Red Flag Check Results ===')
for msg in passed:
    print(f'  PASS: {msg}')
for msg in failed:
    print(f'  FAIL: {msg}')

if failed:
    warnings.warn(f'{len(failed)} red flag(s) triggered â€” investigate before merging')

=== Red Flag Check Results ===
  PASS: BB rate <= 15% for all PAs
  PASS: HR rate â‰¤ 10% for all PAs
  PASS: All PA probability sums â‰ˆ 1.0
  PASS: Cold-start MAE ratio OK (0.53)
  PASS: Model beats naive baseline by 0.59975


## Section 6 â€” Frozen Baseline Summary

**Commit this notebook with cell outputs.**  
Future changes must produce values equal or better in all primary metrics.

In [14]:
# Aggregate brier for convenience
summary_rows = [
    ('Multi-class log-loss',     f'{ll:.5f}',                    '< naive baseline'),
    ('Naive log-loss (uniform)', f'{naive_ll:.5f}',              'â€”'),
    ('Brier HR',                 f'{brier["HR"]:.6f}',           '< 0.038'),
    ('Brier BB',                 f'{brier["BB"]:.6f}',           '< 0.086'),
    ('Brier K',                  f'{brier["K"]:.6f}',            '< 0.160'),
    ('Brier 1B',                 f'{brier["1B"]:.6f}',           '< 0.200'),
    ('Batter week MAE (pts)',    f'{mae:.2f}',                   '-10% vs previous'),
    ('Batter week RMSE (pts)',   f'{rmse:.2f}',                  'â€”'),
    ('Spearman rho (batters)',   f'{rho:.4f}  (p={pval:.3f})',   '> 0.60'),
    ('LR-1 held-out log-loss',   f'{ll_calibrated:.5f}',         '< flat-div baseline'),
    ('LR-1 flat-div log-loss',   f'{ll_flat:.5f}',               'â€”'),
    ('Max batter BB rate',       f'{max(p["BB"] for p in pred_probs):.4f}', '< 0.15'),
    ('Max batter HR rate',       f'{max(p["HR"] for p in pred_probs):.4f}', '< 0.10'),
    ('Sample PAs',               f'{len(pred_probs):,}',         'â‰¥ 4 weeks'),
    ('Red flags triggered',      str(len(failed)),               '0'),
]

summary_df = pd.DataFrame(summary_rows, columns=['Metric', 'Value', 'Target'])
print('=== BASELINE SUMMARY (V2.2) ===')
print(summary_df.to_string(index=False))

=== BASELINE SUMMARY (V2.2) ===
                  Metric             Value              Target
    Multi-class log-loss           1.47969    < naive baseline
Naive log-loss (uniform)           2.07944                 â€”
                Brier HR          0.029400             < 0.038
                Brier BB          0.077776             < 0.086
                 Brier K          0.171461             < 0.160
                Brier 1B          0.121603             < 0.200
   Batter week MAE (pts)              4.38    -10% vs previous
  Batter week RMSE (pts)              5.70                 â€”
  Spearman rho (batters) 0.6000  (p=0.000)              > 0.60
  LR-1 held-out log-loss           1.47145 < flat-div baseline
  LR-1 flat-div log-loss           1.47363                 â€”
      Max batter BB rate            0.1321              < 0.15
      Max batter HR rate            0.0501              < 0.10
              Sample PAs           200,751         â‰¥ 4 weeks
     Red flags triggere